# 01 — FAISS baseline từ CLIPFeatures BTC phát sẵn

**Phụ trách:** Tuấn Anh (N1) · **Mốc:** M1 · **GPU:** KHÔNG cần (CPU đủ)

Đầu vào: thư mục CLIPFeatures của BTC trên Drive chung.
Đầu ra: `faiss_clip.index` + `keyframe_ids.npy` lưu về Drive.

> Mở notebook này từ GitHub: Colab → File → Open notebook → GitHub → dán URL repo.

In [1]:
# === SETUP: mount Drive + cài thư viện ===
from google.colab import drive
drive.mount('/content/drive')
%pip -q install faiss-cpu numpy pandas

# === CONFIG: sửa đường dẫn theo Drive chung của đội ===
DRIVE_ROOT = '/content/drive/MyDrive/AIC2026'   # thư mục chia sẻ chung
CLIP_DIR   = f'{DRIVE_ROOT}/data/clip_features'  # nơi để CLIPFeatures BTC phát
OUT_DIR    = f'{DRIVE_ROOT}/outputs/index_v1_clip'
import os; os.makedirs(OUT_DIR, exist_ok=True)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 65.4 MB/s eta 0:00:00


In [2]:
# === BƯỚC 1: đọc toàn bộ CLIPFeatures ===
# TODO: chỉnh theo format thật của BTC. Thường là 1 file .npy per video,
# tên file = video_id, mỗi dòng = 1 keyframe (thứ tự khớp danh sách keyframe).
import numpy as np, glob, os

feature_files = sorted(glob.glob(f'{CLIP_DIR}/*.npy'))
print(f'{len(feature_files)} file features')
assert feature_files, 'Không thấy file nào — kiểm tra CLIP_DIR'

all_vecs, all_ids = [], []
for fp in feature_files:
    video_id = os.path.splitext(os.path.basename(fp))[0]
    vecs = np.load(fp).astype('float32')
    all_vecs.append(vecs)
    all_ids += [f'{video_id}#{i:06d}' for i in range(len(vecs))]

X = np.concatenate(all_vecs)
print('Tổng keyframe:', X.shape, '— chiều vector:', X.shape[1])
# Chiều vector tiết lộ model: 512=ViT-B/32 hoặc B/16, 768=ViT-L/14 → báo cả đội!

873 file features
Tổng keyframe: (177321, 512) — chiều vector: 512


In [3]:
# === BƯỚC 2: chuẩn hóa L2 + build FAISS IndexFlatIP ===
import faiss
faiss.normalize_L2(X)
index = faiss.IndexFlatIP(X.shape[1])
index.add(X)
print('Index size:', index.ntotal)

faiss.write_index(index, f'{OUT_DIR}/faiss_clip.index')
np.save(f'{OUT_DIR}/keyframe_ids.npy', np.array(all_ids))
print('Đã lưu về', OUT_DIR)

Index size: 177321
Đã lưu về /content/drive/MyDrive/AIC2026/outputs/index_v1_clip


In [4]:
# === BƯỚC 3: smoke test — encode 1 câu text và tìm thử ===
# QUAN TRỌNG: model text PHẢI khớp model BTC dùng tạo CLIPFeatures.
# Thử lần lượt các ứng viên, model nào cho kết quả hợp lý là đúng.
%pip -q install open_clip_torch torch
import open_clip, torch

MODEL_NAME, PRETRAINED = 'ViT-B-32', 'openai'   # TODO: đổi nếu chiều vector không khớp
model, _, _ = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)

q = 'a person eating ice cream near the sea'
with torch.no_grad():
    tv = model.encode_text(tokenizer([q])).float().numpy()
faiss.normalize_L2(tv)
D, I = index.search(tv, 10)
for d, i in zip(D[0], I[0]):
    print(f'{d:.3f}  {all_ids[i]}')
# Mở vài keyframe tương ứng xem có đúng ngữ nghĩa không → nếu lung tung, đổi MODEL_NAME.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00


open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


0.325  L27_V005#000308
0.315  L27_V009#000324
0.314  L27_V009#000322
0.313  L27_V006#000300
0.311  L28_V017#000350
0.311  L27_V005#000076
0.306  L27_V009#000325
0.305  L27_V006#000299
0.303  L28_V017#000349
0.303  L29_V018#000455


In [6]:
import os
data = '/content/drive/MyDrive/AIC2026/data'

# 1. Làm phẳng các thư mục lồng trùng tên (clip_features/clip-features -> clip_features)
for outer in ['clip_features', 'keyframes', 'metadata', 'objects']:
    op = f'{data}/{outer}'
    if not os.path.isdir(op):
        continue
    items = os.listdir(op)
    if len(items) == 1 and os.path.isdir(f'{op}/{items[0]}'):
        inner = f'{op}/{items[0]}'
        tmp = f'{data}/__tmp_{outer}'
        os.rename(inner, tmp)   # nhấc ruột ra ngoài
        os.rmdir(op)            # bỏ vỏ rỗng
        os.rename(tmp, op)      # đặt lại đúng tên chuẩn
        print('✓ đã làm phẳng:', outer)

# 2. Chuẩn hóa tên map-keyframes (gạch ngang theo quy ước của đội)
if os.path.isdir(f'{data}/map_keyframes') and not os.path.isdir(f'{data}/map-keyframes'):
    os.rename(f'{data}/map_keyframes', f'{data}/map-keyframes')
    print('✓ đổi tên map_keyframes -> map-keyframes')

# 3. Kiểm tra lại
print()
for sub in sorted(os.listdir(data)):
    p = f'{data}/{sub}'
    if os.path.isdir(p):
        files = os.listdir(p)
        print(f'{sub:20s} {len(files):7,} mục — ví dụ: {files[:3]}')


clip_features            873 mục — ví dụ: ['L24_V045.npy', 'L24_V019.npy', 'L23_V009.npy']
keyframes                 29 mục — ví dụ: ['L21_V005', 'L21_V008', 'L21_V003']
map-keyframes            882 mục — ví dụ: ['L21_V008.csv', 'L21_V022.csv', 'L22_V009.csv']
metadata                 873 mục — ví dụ: ['L21_V001.json', 'L21_V002.json', 'L21_V003.json']
objects                    0 mục — ví dụ: []
videos                     0 mục — ví dụ: []


In [ ]:
import faiss, torch, numpy as np, open_clip
from PIL import Image
from IPython.display import display

KF_DIR = '/content/drive/MyDrive/AIC2026/data/keyframes'
q = 'a news anchor in a television studio'   # đổi câu tùy thích

def show_top(model_name, k=5):
    model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained='openai')
    tok = open_clip.get_tokenizer(model_name)
    with torch.no_grad():
        tv = model.encode_text(tok([q])).float().numpy()
    faiss.normalize_L2(tv)
    D, I = index.search(tv, k)
    print(f'\n=== {model_name} — "{q}" ===')
    for d, i in zip(D[0], I[0]):
        kf = all_ids[i]; vid, n = kf.split('#')
        print(f'{d:.3f}  {kf}')
        try:
            display(Image.open(f'{KF_DIR}/{vid}/{int(n):03d}.jpg').resize((320, 180)))
        except FileNotFoundError:
            print('   (chưa thấy ảnh — kiểm tra tên file keyframe)')

show_top('ViT-B-32-quickgelu')
show_top('ViT-B-16')